# 1.1 — Calibration: does EM recover known mixture weights from real model samples?

**The README's required first result.** Before fitting anything real, build a synthetic mixture whose
weights we *know*: sample responses from the persona components themselves in fixed proportions, pool
them, score every response under every component, and check that

1. EM over the weights recovers the proportions we used, and
2. the held-out KL estimate $\hat D$ against the true mixture is near zero.

If this fails, nothing downstream means anything. The data come from `scripts/phase1_sample_score.py
--source <name>:<frac>,...` on OLMo 3 base with the description-only components and the `unknown`
framing, i.e. exactly the machinery the real fit uses.

Two synthetic mixtures: a two-component one (`hhh` 0.7 / `evil` 0.3, the README's example) and a
five-component one with unequal weights, which is the harder, more realistic case.

**What "near zero" means here.** For a synthetic pool the true generic distribution *is* the mixture
$\sum_s w^{\text{true}}_s P(\cdot\mid s)$, so we can compute $\log P_0(a_i)$ exactly from the component
scores and the true weights. $\hat D$ then measures only the cost of estimating $w$, and should be
within a couple of standard errors of zero.

In [ ]:
import os, sys, json, textwrap
from pathlib import Path
import numpy as np
from scipy.special import logsumexp
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.mixture import em_weights, kl_estimate, fit_and_evaluate, bootstrap_over_groups, k_sweep, synthetic_mixture_check

RUNS = {  # name -> results directory written by scripts/phase1_sample_score.py
    "hhh0.7_evil0.3": REPO / "results" / "phase1" / "calib_hhh70_evil30",
    "five_unequal":   REPO / "results" / "phase1" / "calib_five",
}
def load(run):
    z = np.load(run / "matrix.npz", allow_pickle=True)
    rows = [json.loads(l) for l in open(run / "rows.jsonl")]
    cfg = json.loads((run / "config.json").read_text())
    return {"L": z["L"], "l0_prompt": z["l0"], "n_tokens": z["n_tokens"], "groups": z["groups"], "source": z["source"].astype(str),
            "personas": list(z["personas"].astype(str)), "rows": rows, "cfg": cfg}
D = {k: load(v) for k, v in RUNS.items() if (v / "matrix.npz").exists()}
for k, d in D.items():
    print(f"{k}: {len(d['rows'])} responses from {len(set(d['groups']))} questions; components {d['personas']}; source counts {dict(zip(*np.unique(d['source'], return_counts=True)))}")

## Weight recovery and KL, full data

In [ ]:
results = {}
for k, d in D.items():
    names = d["personas"]; src_idx = np.array([names.index(s) for s in d["source"]])
    true_w = np.bincount(src_idx, minlength=len(names)) / len(src_idx)
    chk = synthetic_mixture_check(d["L"], src_idx, names)
    l0_true = logsumexp(np.log(np.maximum(true_w, 1e-12))[None, :] + d["L"], axis=1)   # exact generic log-lik for the synthetic pool
    w_em = np.array([chk["em_w"][n] for n in names])
    kl = kl_estimate(l0_true, d["L"], w_em, d["n_tokens"])
    results[k] = {"true_w": true_w, "em_w": w_em, "kl": kl, "confusion": chk["confusion_true_x_pred"]}
    print("=" * 90); print(k)
    print(f"{'component':>10} {'true w':>8} {'EM w':>8}")
    for n, t, e in zip(names, true_w, w_em):
        print(f"{n:>10} {t:8.3f} {e:8.3f}")
    print(f"max |error| = {np.abs(true_w - w_em).max():.3f} | KL(true mixture || fitted) = {kl['kl_per_response']:+.4f} ± {kl['kl_se']:.4f} nats/response ({kl['kl_per_token']:+.5f}/token)")
    print("responsibility argmax vs true source (rows = true, cols = predicted):"); print(chk["confusion"] if "confusion" in chk else chk["confusion_true_x_pred"])

## Held-out KL and bootstrap error bars

Fit on half the questions, evaluate on the other half; bootstrap over questions. This is the exact
procedure the real fit uses, so the numbers here are the calibration floor for Phase 1.

In [ ]:
for k, d in D.items():
    names = d["personas"]; src_idx = np.array([names.index(s) for s in d["source"]])
    true_w = np.bincount(src_idx, minlength=len(names)) / len(src_idx)
    l0_true = logsumexp(np.log(np.maximum(true_w, 1e-12))[None, :] + d["L"], axis=1)
    r = fit_and_evaluate(d["L"], l0_true, d["groups"], d["n_tokens"])
    b = bootstrap_over_groups(d["L"], l0_true, d["groups"], n_boot=200, n_tokens=d["n_tokens"])
    results[k]["heldout"] = r["heldout"]; results[k]["boot_w_sd"] = b["w"].std(0); results[k]["boot_kl"] = b["kl_heldout"]
    print("=" * 90); print(f"{k}: fit on {r['n_train']} responses, evaluate on {r['n_test']}")
    print("  w (split fit):   " + "  ".join(f"{n}={x:.3f}" for n, x in zip(names, r["w"])))
    print("  w bootstrap sd:  " + "  ".join(f"{n}={x:.3f}" for n, x in zip(names, b["w"].std(0))))
    print(f"  held-out KL: {r['heldout']['kl_per_response']:+.4f} ± {r['heldout']['kl_se']:.4f} nats/response; bootstrap: mean {b['kl_heldout'].mean():+.4f}, sd {b['kl_heldout'].std():.4f}")

## K sweep on the synthetic pools

With the true components in the basis, held-out KL should drop to ~0 exactly when all *used*
components are included, and adding unused ones should not help (their weights go to ~0).

In [ ]:
for k, d in D.items():
    names = d["personas"]; src_idx = np.array([names.index(s) for s in d["source"]])
    true_w = np.bincount(src_idx, minlength=len(names)) / len(src_idx)
    l0_true = logsumexp(np.log(np.maximum(true_w, 1e-12))[None, :] + d["L"], axis=1)
    sw = k_sweep(d["L"], l0_true, d["groups"], names, d["n_tokens"])
    best = {}
    for row in sw:
        if row["k"] not in best or row["kl_heldout"] < best[row["k"]]["kl_heldout"]:
            best[row["k"]] = row
    print("=" * 90); print(k)
    for kk, row in sorted(best.items()):
        print(f"  K={kk}: best subset {row['subset']!s:45} held-out KL {row['kl_heldout']:+.3f} ± {row['kl_heldout_se']:.3f}")
    results[k]["ksweep_best"] = best

In [ ]:
fig, axes = plt.subplots(1, len(D), figsize=(5.5 * len(D), 4))
axes = np.atleast_1d(axes)
for ax, (k, d) in zip(axes, D.items()):
    names = d["personas"]; x = np.arange(len(names))
    ax.bar(x - 0.2, results[k]["true_w"], 0.4, color="0.6", label="true")
    ax.bar(x + 0.2, results[k]["em_w"], 0.4, color="#2C6FB3", yerr=results[k]["boot_w_sd"], capsize=3, label="EM ± bootstrap sd")
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("mixture weight"); ax.set_title(k)
    ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.tight_layout(); fig.savefig(REPO / "results" / "phase1" / "1.1_calibration.png", dpi=150); plt.show()
json.dump({k: {"true_w": v["true_w"].tolist(), "em_w": v["em_w"].tolist(), "kl_full": v["kl"], "heldout": v["heldout"],
               "boot_w_sd": v["boot_w_sd"].tolist(), "confusion": np.asarray(v["confusion"]).tolist()} for k, v in results.items()},
          open(REPO / "results" / "phase1" / "1.1_calibration.json", "w"), indent=2)
print("saved results/phase1/1.1_calibration.{png,json}")

## What to look for

- **Weight recovery.** Errors of a few percent are expected from finite samples; errors of tens of
  percent mean the components overlap so much that the weights are not identifiable from ~100-token
  answers, and Phase 1 needs either sharper components or more data.
- **KL.** Should be within ~2 standard errors of zero on held-out questions. Systematically positive
  means the EM fit is losing something (bug, or weights hitting the floor); negative beyond the error
  bar is impossible with the true weights available and would indicate a bug.
- **Confusion matrix.** The off-diagonal mass is the overlap between components on their own samples;
  it is the reason weights have error bars, and it is worth knowing which pairs overlap (expect
  `hhh`/`formal`/`sycophant` to blur more than `evil`).
- **K sweep.** The KL should flatten once the used components are in; unused ones should get ~0 weight.